This part of the code shows GEE part data collection : Multimodal Satalite Data Collection - here we are collecting very improtant data sets for multimodal fusion.

In [ ]:
#  =====================================================
#  ENHANCED GEE PIPELINE FOR REMOTE SENSING FOCUS
#  Version: Agricultural-Only (No SDOH/Weather)
# =====================================================

from google.colab import drive
drive.mount('/content/drive')

import ee, os, time, json, pandas as pd
from google.colab import drive

try:
    ee.Initialize(project='ee-faiz2009cu')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-faiz2009cu')

# ------------------- CONFIG -------------------
TEST_MODE = False
SCALE = 250
FOLDER = 'Agricultural_RS_LE_2025'
BASE = f'/content/drive/MyDrive/{FOLDER}'
os.makedirs(BASE, exist_ok=True)

if TEST_MODE:
    YEARS = [2020, 2021, 2022]
    STATES = ['06']  # California
else:
    YEARS = range(2000, 2025)
    STATES = None

# ------------------- EXPANDED BAND CATALOG -------------------
BANDS = {
    # Your existing sensors
    'landsat89': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
    's2': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12'],
    's1': ['VV', 'VH'],
    'cropland_usda': ['cropland'],
    'dynamic_world': ['water', 'trees', 'grass', 'flooded_vegetation',
                      'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'],
    'water_jrc': ['waterClass'],
    'dem': ['DEM'],

    # NEW: Additional products
    'modis_ndvi': ['NDVI', 'EVI'],
    'modis_lst': ['LST_Day_1km', 'LST_Night_1km'],
    'soil_texture': ['b0'],  # Surface soil texture
    'nightlights': ['avg_rad'],  # Proxy for economic activity (optional)
}

# ------------------- EXPANDED ASSET MAP -------------------
ASSET_MAP = {
    # Existing
    'landsat89':     {'enabled': False, 'id': 'LANDSAT/LC08/C02/T1_L2', 'temporal': 'median', 'add_ndvi': True},
    's2':            {'enabled': False, 'id': 'COPERNICUS/S2_SR_HARMONIZED', 'temporal': 'median', 'add_ndvi': True},
    's1':            {'enabled': False, 'id': 'COPERNICUS/S1_GRD', 'temporal': 'median'},
    'cropland_usda': {'enabled': True, 'id': 'USDA/NASS/CDL', 'temporal': 'mode', 'categorical': True},
    'dynamic_world': {'enabled': True, 'id': 'GOOGLE/DYNAMICWORLD/V1', 'temporal': 'mode', 'categorical': True},
    'water_jrc':     {'enabled': True, 'id': 'JRC/GSW1_4/YearlyHistory', 'temporal': 'mode', 'categorical': True},
    'dem':           {'enabled': True, 'id': 'COPERNICUS/DEM/GLO30', 'static': True},

    # NEW: MODIS for longer time series
    'modis_ndvi': {
        'enabled': True,
        'id': 'MODIS/061/MOD13A1',
        'temporal': 'median',
        'scale_override': 500  # MODIS native resolution
    },

    'modis_lst': {
        'enabled': True,
        'id': 'MODIS/061/MOD11A2',
        'temporal': 'mean',
        'scale_override': 1000,
        'preprocessing': lambda img: img.multiply(0.02).subtract(273.15)  # Kelvin to Celsius
    },

    # NEW: Soil properties (static)
    'soil_texture': {
        'enabled': True,
        'id': 'OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02',
        'static': True
    },

    # OPTIONAL: VIIRS Nightlights (economic activity proxy)
    'nightlights': {
        'enabled': False,  # Set True if you want urban-rural gradient
        'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG',
        'temporal': 'mean'
    }
}

# =====================================================
#               HELPER FUNCTIONS (ADDED FIXES)
# =====================================================

def mask_landsat(image):
    """
    Cloud masking for Landsat 8/9 using QA_PIXEL band.
    Scales optical bands to 0-1 surface reflectance.
    """
    qa = image.select('QA_PIXEL')
    # Mask Dilated Cloud, Cirrus, Cloud, and Cloud Shadow
    mask = qa.bitwiseAnd(1 << 1).eq(0) \
        .And(qa.bitwiseAnd(1 << 2).eq(0)) \
        .And(qa.bitwiseAnd(1 << 3).eq(0)) \
        .And(qa.bitwiseAnd(1 << 4).eq(0))

    # Apply scaling factors for Landsat Collection 2
    return image.updateMask(mask) \
        .multiply(0.0000275).add(-0.2) \
        .copyProperties(image, image.propertyNames())

def mask_s2(image):
    """
    Cloud masking for Sentinel-2 using QA60 band.
    Scales to 0-1 surface reflectance.
    """
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))

    # Scale to 0-1 (S2 data is typically 0-10000)
    return image.updateMask(mask).divide(10000) \
                .copyProperties(image, image.propertyNames())

def fix_cdl(image):
    """Ensure CDL has the correct band name."""
    return image.select(['cropland'])

# =====================================================

# ------------------- TEXTURE FEATURES (NEW) -------------------
def add_texture_features(img):
    """
    Compute GLCM texture metrics for Sentinel-1.
    Useful for crop structure characterization.
    """
    glcm = img.select(['VV']).glcmTexture(size=3)

    texture_bands = [
        'VV_asm',   # Angular Second Moment (uniformity)
        'VV_contrast',
        'VV_corr',  # Correlation
        'VV_ent'    # Entropy (randomness)
    ]

    return img.addBands(glcm.select(texture_bands))

# ------------------- SPECTRAL INDICES (EXPANDED) -------------------
def add_spectral_indices(img, sensor='landsat'):
    """
    Add comprehensive spectral indices for agricultural monitoring.
    """
    if sensor == 'landsat':
        nir, red, green, swir1 = 'SR_B5', 'SR_B4', 'SR_B3', 'SR_B6'
    else:  # sentinel-2
        nir, red, green, swir1 = 'B8', 'B4', 'B3', 'B11'

    # Vegetation indices
    ndvi = img.normalizedDifference([nir, red]).rename('NDVI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': img.select(nir), 'RED': img.select(red), 'BLUE': img.select('SR_B2' if sensor=='landsat' else 'B2')}
    ).rename('EVI')

    savi = img.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': img.select(nir), 'RED': img.select(red)}
    ).rename('SAVI')

    # Moisture/water indices
    ndwi = img.normalizedDifference([green, nir]).rename('NDWI')
    ndmi = img.normalizedDifference([nir, swir1]).rename('NDMI')  # Normalized Difference Moisture Index

    # Soil/bare ground
    bsi = img.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {
            'SWIR': img.select(swir1),
            'RED': img.select(red),
            'NIR': img.select(nir),
            'BLUE': img.select('SR_B2' if sensor=='landsat' else 'B2')
        }
    ).rename('BSI')

    return img.addBands([ndvi, evi, savi, ndwi, ndmi, bsi])

# ------------------- UPDATED PIPELINE -------------------
class AgricultureRSPipeline:
    def __init__(self):
        drive.mount('/content/drive')
        self.fc = ee.FeatureCollection('TIGER/2018/Counties')\
                  .filter(ee.Filter.inList('STATEFP',
                      ['02','15','60','66','69','72','78']).Not())
        self.states = STATES or self.fc.aggregate_array('STATEFP').distinct().getInfo()

    def export(self, table, name, year=None):
        """Export to Drive with error handling."""
        suffix = f"_{year}" if year else ""
        task_name = f"{name}{suffix}"

        task = ee.batch.Export.table.toDrive(
            collection=table,
            folder=FOLDER,
            description=task_name,
            fileNamePrefix=task_name,
            fileFormat='CSV'
        )

        try:
            task.start()
            print(f"→ Task submitted: {task_name}")
            return task
        except ee.EEException as e:
            print(f"!! FAILED: {task_name}. Error: {e}")
            return None

    def run(self):
        """Main execution loop."""
        active_task_count = 0
        QUEUE_LIMIT = 2900

        for asset, cfg in ASSET_MAP.items():
            if not cfg['enabled']:
                continue

            print(f"\n=== {asset.upper()} ===")
            bands = BANDS[asset]
            is_static = cfg.get('static', False)
            scale = cfg.get('scale_override', SCALE)

            for year in (YEARS if not is_static else [None]):
                for state in self.states:
                    # Check existing files
                    suffix = f"_{year}" if year else ""
                    file_path = os.path.join(BASE, f"{asset}_{state}{suffix}.csv")
                    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                        print(f"  -> File exists: {asset}_{state}{suffix}.csv. Skipping.")
                        continue

                    # Queue management
                    while active_task_count >= QUEUE_LIMIT:
                        print(f"  ...Queue full. Waiting 5 min...")
                        time.sleep(300)
                        active_tasks = [t for t in ee.batch.Task.list() if t.state in ['RUNNING', 'READY']]
                        active_task_count = len(active_tasks)

                    # Build image
                    counties = self.fc.filter(ee.Filter.eq('STATEFP', state))
                    geom = counties.geometry().bounds()

                    try:
                        if is_static:
                            img = ee.Image(cfg['id']).select(bands)
                        else:
                            start, end = f'{year}-01-01', f'{year}-12-31'
                            col = ee.ImageCollection(cfg['id']).filterDate(start, end).filterBounds(geom)

                            # Sensor-specific preprocessing
                            if asset == 'landsat89':
                                if year >= 2022:
                                    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(start, end).filterBounds(geom)
                                    col = col.merge(l9)
                                # --- ERROR FIXED: mask_landsat was missing ---
                                col = col.map(mask_landsat)
                                col = col.map(lambda i: add_spectral_indices(i, 'landsat'))

                            elif asset == 's2':
                                # --- ERROR FIXED: mask_s2 was missing ---
                                col = col.map(mask_s2)
                                col = col.map(lambda i: add_spectral_indices(i, 's2'))

                            elif asset == 's1':
                                col = col.map(add_texture_features)

                            elif asset == 'modis_lst' and 'preprocessing' in cfg:
                                col = col.map(cfg['preprocessing'])

                            elif asset == 'cropland_usda':
                                # --- ERROR FIXED: fix_cdl was missing ---
                                col = col.map(fix_cdl)

                            # Temporal reduction
                            if cfg['temporal'] == 'median':
                                img = col.median()
                            elif cfg['temporal'] == 'mean':
                                img = col.mean()
                            else:
                                img = col.mode()

                            # Select final bands
                            all_bands = bands + (['NDVI', 'EVI', 'SAVI', 'NDWI', 'NDMI', 'BSI'] if cfg.get('add_ndvi') else [])
                            if asset == 's1':
                                all_bands += ['VV_asm', 'VV_contrast', 'VV_corr', 'VV_ent']

                            img = img.select([b for b in all_bands if b in img.bandNames().getInfo()])

                        # Reducers
                        red = ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True)
                        if cfg.get('categorical'):
                            red = red.combine(ee.Reducer.frequencyHistogram(), '', True)
                        else:
                            red = red.combine(ee.Reducer.percentile([10, 25, 50, 75, 90]), '', True)

                        # Spatial aggregation
                        stats = img.reduceRegions(
                            collection=counties,
                            reducer=red,
                            scale=scale,
                            tileScale=4
                        )

                        # Tag and export
                        def tag(f):
                            props = {'STATEFP': f.get('STATEFP'), 'COUNTYFP': f.get('COUNTYFP'), 'NAME': f.get('NAME')}
                            if year: props['year'] = year
                            return ee.Feature(None, props).copyProperties(f, f.propertyNames())

                        stats = stats.map(tag)
                        new_task = self.export(stats, f"{asset}_{state}", year)
                        if new_task:
                            active_task_count += 1

                    except Exception as e:
                        print(f"!! ERROR: {asset}_{state}_{year}: {e}")

        print(f"\nAll tasks queued! → Drive → {FOLDER}")

# RUN
pipeline = AgricultureRSPipeline()
pipeline.run()

In [ ]:
#  =====================================================
#  ENHANCED GEE PIPELINE FOR REMOTE SENSING FOCUS
#  Version: Agricultural-Only (No SDOH/Weather)
# =====================================================

# ------------------- CONFIG -------------------
TEST_MODE = False
SCALE = 250
FOLDER = 'Agricultural_RS_LE_2025'
BASE = f'/content/drive/MyDrive/{FOLDER}'
os.makedirs(BASE, exist_ok=True)

if TEST_MODE:
    YEARS = [2020, 2021, 2022]
    STATES = ['06']  # California
else:
    YEARS = range(2000, 2025)
    STATES = None

# ------------------- EXPANDED BAND CATALOG -------------------
BANDS = {
    # Your existing sensors
    'landsat89': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
    's2': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12'],
    's1': ['VV', 'VH'],
    'cropland_usda': ['cropland'],
    'dynamic_world': ['water', 'trees', 'grass', 'flooded_vegetation',
                      'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice'],
    'water_jrc': ['waterClass'],
    'dem': ['DEM'],

    # NEW: Additional products
    'modis_ndvi': ['NDVI', 'EVI'],
    'modis_lst': ['LST_Day_1km', 'LST_Night_1km'],
    'soil_texture': ['b0'],  # Surface soil texture
    'nightlights': ['avg_rad'],  # Proxy for economic activity (optional)
}

# ------------------- EXPANDED ASSET MAP -------------------
ASSET_MAP = {
    # Existing
    'landsat89':     {'enabled': True, 'id': 'LANDSAT/LC08/C02/T1_L2', 'temporal': 'median', 'add_ndvi': True},
    's2':            {'enabled': True, 'id': 'COPERNICUS/S2_SR_HARMONIZED', 'temporal': 'median', 'add_ndvi': True},
    's1':            {'enabled': True, 'id': 'COPERNICUS/S1_GRD', 'temporal': 'median'},
    'cropland_usda': {'enabled': False, 'id': 'USDA/NASS/CDL', 'temporal': 'mode', 'categorical': True},
    'dynamic_world': {'enabled': False, 'id': 'GOOGLE/DYNAMICWORLD/V1', 'temporal': 'mode', 'categorical': True},
    'water_jrc':     {'enabled': False, 'id': 'JRC/GSW1_4/YearlyHistory', 'temporal': 'mode', 'categorical': True},
    'dem':           {'enabled': False, 'id': 'COPERNICUS/DEM/GLO30', 'static': True},

    # NEW: MODIS for longer time series
    'modis_ndvi': {
        'enabled': True,
        'id': 'MODIS/061/MOD13A1',
        'temporal': 'median',
        'scale_override': 500  # MODIS native resolution
    },

    'modis_lst': {
        'enabled': True,
        'id': 'MODIS/061/MOD11A2',
        'temporal': 'mean',
        'scale_override': 1000,
        'preprocessing': lambda img: img.multiply(0.02).subtract(273.15)  # Kelvin to Celsius
    },

    # NEW: Soil properties (static)
    'soil_texture': {
        'enabled': True,
        'id': 'OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02',
        'static': True
    },

    # OPTIONAL: VIIRS Nightlights (economic activity proxy)
    'nightlights': {
        'enabled': False,  # Set True if you want urban-rural gradient
        'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG',
        'temporal': 'mean'
    }
}

# =====================================================
#               HELPER FUNCTIONS (ADDED FIXES)
# =====================================================

def mask_landsat(image):
    """
    Cloud masking for Landsat 8/9 using QA_PIXEL band.
    Scales optical bands to 0-1 surface reflectance.
    """
    qa = image.select('QA_PIXEL')
    # Mask Dilated Cloud, Cirrus, Cloud, and Cloud Shadow
    mask = qa.bitwiseAnd(1 << 1).eq(0) \
        .And(qa.bitwiseAnd(1 << 2).eq(0)) \
        .And(qa.bitwiseAnd(1 << 3).eq(0)) \
        .And(qa.bitwiseAnd(1 << 4).eq(0))

    # Apply scaling factors for Landsat Collection 2
    return image.updateMask(mask) \
        .multiply(0.0000275).add(-0.2) \
        .copyProperties(image, image.propertyNames())

def mask_s2(image):
    """
    Cloud masking for Sentinel-2 using QA60 band.
    Scales to 0-1 surface reflectance.
    """
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))

    # Scale to 0-1 (S2 data is typically 0-10000)
    return image.updateMask(mask).divide(10000) \
                .copyProperties(image, image.propertyNames())

def fix_cdl(image):
    """Ensure CDL has the correct band name."""
    return image.select(['cropland'])

# =====================================================

# ------------------- TEXTURE FEATURES (NEW) -------------------
def add_texture_features(img):
    """
    Compute GLCM texture metrics for Sentinel-1.
    Useful for crop structure characterization.
    """
    glcm = img.select(['VV']).glcmTexture(size=3)

    texture_bands = [
        'VV_asm',   # Angular Second Moment (uniformity)
        'VV_contrast',
        'VV_corr',  # Correlation
        'VV_ent'    # Entropy (randomness)
    ]

    return img.addBands(glcm.select(texture_bands))

# ------------------- SPECTRAL INDICES (EXPANDED) -------------------
def add_spectral_indices(img, sensor='landsat'):
    """
    Add comprehensive spectral indices for agricultural monitoring.
    """
    if sensor == 'landsat':
        nir, red, green, swir1 = 'SR_B5', 'SR_B4', 'SR_B3', 'SR_B6'
    else:  # sentinel-2
        nir, red, green, swir1 = 'B8', 'B4', 'B3', 'B11'

    # Vegetation indices
    ndvi = img.normalizedDifference([nir, red]).rename('NDVI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': img.select(nir), 'RED': img.select(red), 'BLUE': img.select('SR_B2' if sensor=='landsat' else 'B2')}
    ).rename('EVI')

    savi = img.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': img.select(nir), 'RED': img.select(red)}
    ).rename('SAVI')

    # Moisture/water indices
    ndwi = img.normalizedDifference([green, nir]).rename('NDWI')
    ndmi = img.normalizedDifference([nir, swir1]).rename('NDMI')  # Normalized Difference Moisture Index

    # Soil/bare ground
    bsi = img.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {
            'SWIR': img.select(swir1),
            'RED': img.select(red),
            'NIR': img.select(nir),
            'BLUE': img.select('SR_B2' if sensor=='landsat' else 'B2')
        }
    ).rename('BSI')

    return img.addBands([ndvi, evi, savi, ndwi, ndmi, bsi])

# ------------------- UPDATED PIPELINE -------------------
class AgricultureRSPipeline:
    def __init__(self):
        drive.mount('/content/drive')
        self.fc = ee.FeatureCollection('TIGER/2018/Counties')\
                  .filter(ee.Filter.inList('STATEFP',
                      ['02','15','60','66','69','72','78']).Not())
        self.states = STATES or self.fc.aggregate_array('STATEFP').distinct().getInfo()

    def export(self, table, name, year=None):
        """Export to Drive with error handling."""
        suffix = f"_{year}" if year else ""
        task_name = f"{name}{suffix}"

        task = ee.batch.Export.table.toDrive(
            collection=table,
            folder=FOLDER,
            description=task_name,
            fileNamePrefix=task_name,
            fileFormat='CSV'
        )

        try:
            task.start()
            print(f"→ Task submitted: {task_name}")
            return task
        except ee.EEException as e:
            print(f"!! FAILED: {task_name}. Error: {e}")
            return None

    def run(self):
        """Main execution loop."""
        active_task_count = 0
        QUEUE_LIMIT = 2900

        for asset, cfg in ASSET_MAP.items():
            if not cfg['enabled']:
                continue

            print(f"\n=== {asset.upper()} ===")
            bands = BANDS[asset]
            is_static = cfg.get('static', False)
            scale = cfg.get('scale_override', SCALE)

            for year in (YEARS if not is_static else [None]):
                for state in self.states:
                    # Check existing files
                    suffix = f"_{year}" if year else ""
                    file_path = os.path.join(BASE, f"{asset}_{state}{suffix}.csv")
                    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                        print(f"  -> File exists: {asset}_{state}{suffix}.csv. Skipping.")
                        continue

                    # Queue management
                    while active_task_count >= QUEUE_LIMIT:
                        print(f"  ...Queue full. Waiting 5 min...")
                        time.sleep(300)
                        active_tasks = [t for t in ee.batch.Task.list() if t.state in ['RUNNING', 'READY']]
                        active_task_count = len(active_tasks)

                    # Build image
                    counties = self.fc.filter(ee.Filter.eq('STATEFP', state))
                    geom = counties.geometry().bounds()

                    try:
                        if is_static:
                            img = ee.Image(cfg['id']).select(bands)
                        else:
                            start, end = f'{year}-01-01', f'{year}-12-31'
                            col = ee.ImageCollection(cfg['id']).filterDate(start, end).filterBounds(geom)

                            # Sensor-specific preprocessing
                            if asset == 'landsat89':
                                if year >= 2022:
                                    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(start, end).filterBounds(geom)
                                    col = col.merge(l9)
                                # --- ERROR FIXED: mask_landsat was missing ---
                                col = col.map(mask_landsat)
                                col = col.map(lambda i: add_spectral_indices(i, 'landsat'))

                            elif asset == 's2':
                                # --- ERROR FIXED: mask_s2 was missing ---
                                col = col.map(mask_s2)
                                col = col.map(lambda i: add_spectral_indices(i, 's2'))

                            elif asset == 's1':
                                col = col.map(add_texture_features)

                            elif asset == 'modis_lst' and 'preprocessing' in cfg:
                                col = col.map(cfg['preprocessing'])

                            elif asset == 'cropland_usda':
                                # --- ERROR FIXED: fix_cdl was missing ---
                                col = col.map(fix_cdl)

                            # Temporal reduction
                            if cfg['temporal'] == 'median':
                                img = col.median()
                            elif cfg['temporal'] == 'mean':
                                img = col.mean()
                            else:
                                img = col.mode()

                            # Select final bands
                            all_bands = bands + (['NDVI', 'EVI', 'SAVI', 'NDWI', 'NDMI', 'BSI'] if cfg.get('add_ndvi') else [])
                            if asset == 's1':
                                all_bands += ['VV_asm', 'VV_contrast', 'VV_corr', 'VV_ent']

                            img = img.select([b for b in all_bands if b in img.bandNames().getInfo()])

                        # Reducers
                        red = ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True)
                        if cfg.get('categorical'):
                            red = red.combine(ee.Reducer.frequencyHistogram(), '', True)
                        else:
                            red = red.combine(ee.Reducer.percentile([10, 25, 50, 75, 90]), '', True)

                        # Spatial aggregation
                        stats = img.reduceRegions(
                            collection=counties,
                            reducer=red,
                            scale=scale,
                            tileScale=4
                        )

                        # Tag and export
                        def tag(f):
                            props = {'STATEFP': f.get('STATEFP'), 'COUNTYFP': f.get('COUNTYFP'), 'NAME': f.get('NAME')}
                            if year: props['year'] = year
                            return ee.Feature(None, props).copyProperties(f, f.propertyNames())

                        stats = stats.map(tag)
                        new_task = self.export(stats, f"{asset}_{state}", year)
                        if new_task:
                            active_task_count += 1

                    except Exception as e:
                        print(f"!! ERROR: {asset}_{state}_{year}: {e}")

        print(f"\nAll tasks queued! → Drive → {FOLDER}")

# RUN
pipeline = AgricultureRSPipeline()
pipeline.run()

s1 Data collection

In [ ]:
# ------------------- CONFIG -------------------
TEST_MODE = False
SCALE = 250
FOLDER = 'Agricultural_RS_LE_2025'
BASE = f'/content/drive/MyDrive/{FOLDER}'
os.makedirs(BASE, exist_ok=True)

if TEST_MODE:
    YEARS = range(2020, 2021) # Reduced for testing
    STATES = ['06']  # California
else:
    YEARS = range(2000, 2025)
    STATES = None

# ------------------- BAND CATALOG -------------------
BANDS = {
    'landsat89': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
    's2': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12'],
    's1': ['VV', 'VH'], # We will strictly enforce VV/VH to prevent errors
}

# ------------------- ASSET MAP -------------------
ASSET_MAP = {
    'landsat89': {'enabled': False, 'id': 'LANDSAT/LC08/C02/T1_L2', 'temporal': 'median', 'add_ndvi': True},
    's2': {'enabled': False, 'id': 'COPERNICUS/S2_SR_HARMONIZED', 'temporal': 'median', 'add_ndvi': True},
    's1': {'enabled': True, 'id': 'COPERNICUS/S1_GRD', 'temporal': 'median'},
    'cropland_usda': {'enabled': False, 'id': 'USDA/NASS/CDL', 'temporal': 'mode', 'categorical': True},
    'dynamic_world': {'enabled': False, 'id': 'GOOGLE/DYNAMICWORLD/V1', 'temporal': 'mode', 'categorical': True},
    'water_jrc': {'enabled': False, 'id': 'JRC/GSW1_4/YearlyHistory', 'temporal': 'mode', 'categorical': True},
    'dem': {'enabled': False, 'id': 'COPERNICUS/DEM/GLO30', 'static': True},
    'modis_ndvi': {'enabled': False, 'id': 'MODIS/061/MOD13A1', 'temporal': 'median', 'scale_override': 500},
    'modis_lst': {'enabled': False, 'id': 'MODIS/061/MOD11A2', 'temporal': 'mean', 'scale_override': 1000, 'preprocessing': lambda img: img.multiply(0.02).subtract(273.15)},
    'nightlights': {'enabled': False, 'id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG', 'temporal': 'mean'}
}

# ------------------- HELPER FUNCTIONS -------------------
def mask_landsat(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 1).eq(0) \
        .And(qa.bitwiseAnd(1 << 2).eq(0)) \
        .And(qa.bitwiseAnd(1 << 3).eq(0)) \
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(mask) \
        .multiply(0.0000275).add(-0.2) \
        .copyProperties(image, image.propertyNames())

def mask_s2(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
                 qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000) \
        .copyProperties(image, image.propertyNames())

def add_spectral_indices(img, sensor='landsat'):
    if sensor == 'landsat':
        nir, red = 'SR_B5', 'SR_B4'
    else:
        nir, red = 'B8', 'B4'

    ndvi = img.normalizedDifference([nir, red]).rename('NDVI')
    return img.addBands([ndvi])

# ------------------- ROBUST S1 TEXTURE FUNCTION -------------------
def add_s1_texture(img):
    """
    Computes GLCM on VV band.
    Assumes input image has 'VV' band (guaranteed by filter in main loop).
    """
    # 1. Select VV band
    vv = img.select('VV')

    # 2. Fix for "Only 32-bit integers":
    # Scale float (-20 to 0) to integer (-2000 to 0) so GLCM works.
    # .toInt32() is CRITICAL here.
    vv_int = vv.multiply(100).toInt32()

    # 3. Compute GLCM
    # Neighborhood size 3 is standard/efficient
    glcm = vv_int.glcmTexture(size=3)

    # 4. Select only the useful metrics and rename them cleanly
    # Default names are usually 'VV_asm', 'VV_contrast' etc.
    bands_to_keep = ['VV_asm', 'VV_contrast', 'VV_corr', 'VV_ent']

    return img.addBands(glcm.select(bands_to_keep))

# ------------------- PIPELINE CLASS -------------------
class AgricultureRSPipeline:
    def __init__(self):
        try:
            drive.mount('/content/drive')
        except:
            pass
        # Exclude territories for cleaner US data
        self.fc = ee.FeatureCollection('TIGER/2018/Counties') \
            .filter(ee.Filter.inList('STATEFP', ['02', '15', '60', '66', '69', '72', '78']).Not())

        if STATES:
             self.states = STATES
        else:
             # Get list of states dynamically
             self.states = self.fc.aggregate_array('STATEFP').distinct().getInfo()

    def export(self, table, name, year=None):
        """Export to Drive with error handling."""
        suffix = f"_{year}" if year else ""
        task_name = f"{name}{suffix}"

        task = ee.batch.Export.table.toDrive(
            collection=table,
            folder=FOLDER,
            description=task_name,
            fileNamePrefix=task_name,
            fileFormat='CSV'
        )

        try:
            task.start()
            print(f"→ Task submitted: {task_name}")
            return task
        except ee.EEException as e:
            print(f"!! FAILED: {task_name}. Error: {e}")
            return None

    def run(self):
        """Main execution loop."""
        active_task_count = 0

        for asset, cfg in ASSET_MAP.items():
            if not cfg['enabled']:
                continue

            print(f"\n=== {asset.upper()} ===")
            bands = BANDS.get(asset, [])
            is_static = cfg.get('static', False)
            scale = cfg.get('scale_override', SCALE)

            for year in (YEARS if not is_static else [None]):
                for state in self.states:
                    # Check existing files to skip processing
                    suffix = f"_{year}" if year else ""
                    file_path = os.path.join(BASE, f"{asset}_{state}{suffix}.csv")
                    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                        print(f"  -> File exists: {asset}_{state}{suffix}.csv. Skipping.")
                        continue

                    # Define ROI
                    counties = self.fc.filter(ee.Filter.eq('STATEFP', state))
                    geom = counties.geometry().bounds()

                    try:
                        if is_static:
                            img = ee.Image(cfg['id']).select(bands)
                        else:
                            start, end = f'{year}-01-01', f'{year}-12-31'
                            col = ee.ImageCollection(cfg['id']).filterDate(start, end).filterBounds(geom)

                            # --- SENSOR SPECIFIC PROCESSING ---
                            if asset == 'landsat89':
                                if year >= 2022:
                                    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(start, end).filterBounds(geom)
                                    col = col.merge(l9)
                                col = col.map(mask_landsat).map(lambda i: add_spectral_indices(i, 'landsat'))

                            elif asset == 's2':
                                col = col.map(mask_s2).map(lambda i: add_spectral_indices(i, 's2'))

                            elif asset == 's1':
                                # --- FIX: PRE-FILTERING IS KEY ---
                                # Filter for IW mode (standard over land)
                                col = col.filter(ee.Filter.eq('instrumentMode', 'IW'))
                                # Filter for images that actually have VV and VH bands
                                col = col.filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                                col = col.filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                                # Direction Pass (Ascending/Descending mix can cause noise, but usually handled by median)
                                # Now map the texture function safely
                                col = col.map(add_s1_texture)

                            elif asset == 'modis_lst' and 'preprocessing' in cfg:
                                col = col.map(cfg['preprocessing'])

                            elif asset == 'cropland_usda':
                                def fix_cdl(image): return image.select(['cropland'])
                                col = col.map(fix_cdl)

                            # Check if collection is empty after filtering
                            # (We use .first() to check without downloading everything)
                            if col.size().getInfo() == 0:
                                print(f"  !! No data found for {asset} in state {state} for {year}")
                                continue

                            # Temporal reduction
                            if cfg['temporal'] == 'median':
                                img = col.median()
                            elif cfg['temporal'] == 'mean':
                                img = col.mean()
                            else:
                                img = col.mode()

                            # --- DYNAMIC BAND SELECTION ---
                            # We construct the list of bands we expect to be present
                            expected_bands = list(bands)
                            if cfg.get('add_ndvi'):
                                expected_bands += ['NDVI']
                            if asset == 's1':
                                # Add the texture bands we created
                                expected_bands += ['VV_asm', 'VV_contrast', 'VV_corr', 'VV_ent']

                            # Select only bands that actually exist in the composite
                            available_bands = img.bandNames()
                            img = img.select(available_bands.filter(ee.Filter.inList('item', expected_bands)))

                        # Reducers
                        red = ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True)
                        if cfg.get('categorical'):
                            red = red.combine(ee.Reducer.frequencyHistogram(), '', True)
                        else:
                            red = red.combine(ee.Reducer.percentile([10, 25, 50, 75, 90]), '', True)

                        # Spatial aggregation
                        stats = img.reduceRegions(
                            collection=counties,
                            reducer=red,
                            scale=scale,
                            tileScale=4 # Increased tileScale for heavy S1 computation
                        )

                        # Tag and export
                        def tag(f):
                            props = {'STATEFP': f.get('STATEFP'), 'COUNTYFP': f.get('COUNTYFP'), 'NAME': f.get('NAME')}
                            if year: props['year'] = year
                            return ee.Feature(None, props).copyProperties(f, f.propertyNames())

                        stats = stats.map(tag)

                        # Final check: Don't export empty results
                        new_task = self.export(stats, f"{asset}_{state}", year)
                        if new_task:
                            active_task_count += 1

                    except Exception as e:
                        print(f"!! ERROR processing {asset} for State {state} in {year}: {e}")

        print(f"\nAll tasks queued! → Drive → {FOLDER}")

# RUN
pipeline = AgricultureRSPipeline()
pipeline.run()

Data collection finished here
Merge code starts here
Merge code to individual master files; with normaliztion


This is the output of merging code that I have run in the colab env. 

🚜 Processing CROPLAND_USDA...
Reading Cropland_USDA: 100%|██████████| 1096/1096 [01:47<00:00, 10.24it/s] 
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Cropland_USDA_vFinal.csv
   Shape: (73847, 153) (Rows, Cols)
   Years: 2000 to 2024

🚜 Processing DYNAMIC_WORLD...
Reading Dynamic_World: 100%|██████████| 104/104 [02:23<00:00,  1.38s/it]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Dynamic_World_vFinal.csv
   Shape: (1348, 36) (Rows, Cols)
   Years: 2015 to 2024

🚜 Processing WATER_JRC...
Reading Water_JRC: 100%|██████████| 1078/1078 [00:08<00:00, 124.71it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Water_JRC_vFinal.csv
   Shape: (68376, 24) (Rows, Cols)
   Years: 2000 to 2021

🚜 Processing LANDSAT...
Reading Landsat: 100%|██████████| 588/588 [00:05<00:00, 111.61it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Landsat_vFinal.csv
   Shape: (37296, 102) (Rows, Cols)
   Years: 2013 to 2024

🚜 Processing SENTINEL2...
Reading Sentinel2: 100%|██████████| 229/229 [00:02<00:00, 108.27it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Sentinel2_vFinal.csv
   Shape: (12002, 102) (Rows, Cols)
   Years: 2015 to 2024

🚜 Processing MODIS_NDVI...
Reading MODIS_NDVI: 100%|██████████| 1225/1225 [00:05<00:00, 212.93it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_MODIS_NDVI_vFinal.csv
   Shape: (77700, 32) (Rows, Cols)
   Years: 2000 to 2024

🚜 Processing MODIS_LST...
Reading MODIS_LST: 100%|██████████| 1225/1225 [00:05<00:00, 205.01it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_MODIS_LST_vFinal.csv
   Shape: (77700, 32) (Rows, Cols)
   Years: 2000 to 2024

🚜 Processing DEM...
Reading DEM: 100%|██████████| 45/45 [00:00<00:00, 182.34it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_DEM_vFinal.csv
   Shape: (2782, 24) (Rows, Cols)

🚜 Processing SOIL...
Reading Soil: 100%|██████████| 49/49 [00:41<00:00,  1.19it/s]
✅ Saved: /content/drive/MyDrive/Agricultural_RS_LE_2025/master csv/MASTER_Soil_vFinal.csv
   Shape: (3108, 24) (Rows, Cols)

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION & CLASS MAPPINGS
# ==========================================
SOURCE_DIR = '/content/drive/MyDrive/Agricultural_RS_LE_2025'
DEST_DIR = os.path.join(SOURCE_DIR, 'master csv')

if not os.path.exists(DEST_DIR):
    os.makedirs(DEST_DIR)

# --- CLASS DEFINITIONS (The Decoder Rings) ---
CLASS_MAPS = {
    'Cropland_USDA': {
        '1': 'Corn', '2': 'Cotton', '3': 'Rice', '4': 'Sorghum', '5': 'Soybeans',
        '6': 'Sunflower', '10': 'Peanuts', '11': 'Tobacco', '12': 'Sweet Corn',
        '21': 'Barley', '23': 'Spring Wheat', '24': 'Winter Wheat', '28': 'Oats',
        '36': 'Alfalfa', '37': 'Other Hay', '44': 'Other Crops', '61': 'Fallow',
        '111': 'Open Water', '121': 'Dev_Open', '122': 'Dev_Low',
        '123': 'Dev_Med', '124': 'Dev_High', '141': 'Forest_Deciduous',
        '142': 'Forest_Evergreen', '143': 'Forest_Mixed', '152': 'Shrubland',
        '176': 'Grassland', '190': 'Wetlands_Woody', '195': 'Wetlands_Herbaceous'
    },
    'Dynamic_World': {
        '0': 'Water', '1': 'Trees', '2': 'Grass', '3': 'FloodedVeg',
        '4': 'Crops', '5': 'ShrubScrub', '6': 'Built', '7': 'Bare', '8': 'SnowIce'
    },
    'Water_JRC': {
        '1': 'NotWater', '2': 'Seasonal', '3': 'Permanent'
    }
}

# ==========================================
# 2. PARSING ENGINE
# ==========================================
def parse_histogram_str(text):
    """Robustly parses string '{k=v, ...}' or JSON into a python dict."""
    if pd.isna(text) or str(text).strip() in ['', '{}', 'nan']:
        return {}

    # 1. Fast Path: Standard JSON
    if ':' in str(text) and '{' in str(text):
        try:
            return ast.literal_eval(str(text))
        except: pass

    # 2. Robust Path: GEE "key=value" format
    clean = str(text).replace('{', '').replace('}', '')
    if not clean: return {}

    # Regex handles scientific notation (e.g. 1.2e-5)
    pattern = r'([\d\w\.\-\+]+)\s*=\s*([\d\w\.\-\+eE]+)'
    matches = re.findall(pattern, clean)

    data = {}
    for k, v in matches:
        try:
            # Clean Key: Convert "1.0" -> "1" (Int) for cleaner mapping
            key_num = float(k)
            key = str(int(key_num)) if key_num.is_integer() else str(k)

            data[key] = float(v)
        except: continue
    return data

# ==========================================
# 3. GENERIC PROCESSING (One Code to Rule Them All)
# ==========================================
def process_sensor_master(sensor_name, file_pattern):
    print(f"\n🚜 Processing {sensor_name.upper()}...")

    files = glob.glob(os.path.join(SOURCE_DIR, file_pattern))
    if not files:
        print(f"   ⚠️ No files found for {sensor_name}")
        return

    all_rows = []

    # Get specific mapping for this sensor (if any)
    sensor_map = CLASS_MAPS.get(sensor_name, {})

    for f in tqdm(files, desc=f"Reading {sensor_name}"):
        try:
            # 1. Load File
            df = pd.read_csv(f, on_bad_lines='skip', low_memory=False)

            # 2. Metadata Cleanup
            if 'system:index' in df.columns:
                df = df.drop(columns=['system:index', '.geo', 'Unnamed: 0'], errors='ignore')

            # Ensure GEOID is padded string
            if 'GEOID' not in df.columns and 'STATEFP' in df.columns:
                 df['GEOID'] = df['STATEFP'].astype(str).str.zfill(2) + \
                               df['COUNTYFP'].astype(str).str.zfill(3)

            # Recover Year if missing
            if 'year' not in df.columns:
                parts = os.path.basename(f).replace('.csv','').split('_')
                if parts[-1].isdigit() and len(parts[-1]) == 4:
                    df['year'] = int(parts[-1])

            # 3. HISTOGRAM EXPLOSION
            # Find any column ending in 'histogram'
            hist_cols = [c for c in df.columns if 'histogram' in c.lower()]

            if hist_cols:
                for h_col in hist_cols:
                    # Skip Dynamic World "Confidence" histograms (Probability noise)
                    # We only want the Class Counts (Integer keys)
                    sample = df[h_col].dropna().iloc[0] if not df[h_col].dropna().empty else ""
                    if '0.1' in str(sample) and sensor_name == 'Dynamic_World':
                        df = df.drop(columns=[h_col]) # Drop noise
                        continue

                    # Parse
                    parsed_series = df[h_col].apply(parse_histogram_str)

                    # Explode
                    exploded = pd.json_normalize(parsed_series)
                    exploded.index = df.index

                    # --- INTELLIGENT RENAMING ---
                    new_names = {}
                    for col in exploded.columns:
                        # col is likely "1", "5", etc.
                        if col in sensor_map:
                            # Map "1" -> "Corn"
                            name_suffix = sensor_map[col]
                        else:
                            # Fallback "Class_1"
                            name_suffix = f"Class_{col}"

                        new_names[col] = f"{sensor_name}_{name_suffix}_pct"

                    exploded = exploded.rename(columns=new_names)

                    # --- CRITICAL: ZERO FILLING (LOCAL ONLY) ---
                    # Only fill 0s for crops *within this file*.
                    # If this file exists, we know missing keys = 0 pixels.
                    exploded = exploded.fillna(0)

                    # Normalize to Percentage
                    row_sums = exploded.sum(axis=1)
                    row_sums[row_sums == 0] = 1
                    exploded = exploded.div(row_sums, axis=0)

                    # Merge & Clean
                    df = pd.concat([df, exploded], axis=1)
                    df = df.drop(columns=[h_col])

            all_rows.append(df)

        except Exception as e:
            pass

    # 4. GRAND MERGE (PRESERVING NANs)
    if all_rows:
        # Stack all chunks
        master_df = pd.concat(all_rows, axis=0, ignore_index=True, sort=False)

        # --- CRITICAL: VACANT CELL HANDLING ---
        # We do NOT run master_df.fillna(0) globally.
        # This ensures that if a Year/County didn't exist in the input files,
        # its columns remain NaN (Vacant), indicating "No Data Collected".

        # Final ID Polish
        if 'GEOID' in master_df.columns:
            master_df['GEOID'] = master_df['GEOID'].astype(str).str.split('.').str[0].str.zfill(5)

        # Sort
        sort_cols = [c for c in ['GEOID', 'year'] if c in master_df.columns]
        if sort_cols:
            master_df = master_df.sort_values(sort_cols)

        # Save
        out_path = os.path.join(DEST_DIR, f'MASTER_{sensor_name}_vFinal.csv')
        master_df.to_csv(out_path, index=False)
        print(f"✅ Saved: {out_path}")
        print(f"   Shape: {master_df.shape} (Rows, Cols)")

        # Audit
        if 'year' in master_df.columns:
            years = sorted(master_df['year'].dropna().unique().astype(int))
            print(f"   Years: {min(years)} to {max(years)}")

# ==========================================
# 4. EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    # Define datasets to process
    DATASETS = {
        'Cropland_USDA': 'cropland_usda_*.csv',
        'Dynamic_World': 'dynamic_world_*.csv',
        'Water_JRC':     'water_jrc_*.csv',
        'Landsat':       'landsat89_*.csv',
        'Sentinel2':     's2_*.csv',
        'MODIS_NDVI':    'modis_ndvi_*.csv',
        'MODIS_LST':     'modis_lst_*.csv',
        'DEM':           'dem_*.csv',
        'Soil':          'soil_texture_*.csv'
    }

    for name, pattern in DATASETS.items():
        process_sensor_master(name, pattern)

Merging S1 missed in above code

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
SOURCE_DIR = '/content/drive/MyDrive/Agricultural_RS_LE_2025'
DEST_DIR = os.path.join(SOURCE_DIR, 'master csv')

if not os.path.exists(DEST_DIR):
    os.makedirs(DEST_DIR)

# ==========================================
# 2. SENTINEL-1 PROCESSOR
# ==========================================
def process_sentinel1_master():
    print(f"\n🛰️  Processing SENTINEL-1 (SAR Structure)...")

    # Pattern to match your files (e.g., s1_01_2015.csv)
    files = glob.glob(os.path.join(SOURCE_DIR, 's1_*.csv'))

    if not files:
        print("   ⚠️ No Sentinel-1 files found (checked 's1_*.csv').")
        return

    print(f"   -> Found {len(files)} file chunks. Stacking...")

    all_rows = []

    for f in tqdm(files, desc="Reading S1"):
        try:
            # 1. Load File (Low memory mode false to prevent type errors)
            df = pd.read_csv(f, on_bad_lines='skip', low_memory=False)

            # 2. Metadata Cleanup
            # Drop GEE system columns if they exist
            cols_to_drop = ['system:index', '.geo', 'Unnamed: 0']
            df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

            # 3. Ensure GEOID is correct
            # If GEOID is missing, try to build it from State/County FP
            if 'GEOID' not in df.columns and 'STATEFP' in df.columns:
                 df['GEOID'] = df['STATEFP'].astype(str).str.zfill(2) + \
                               df['COUNTYFP'].astype(str).str.zfill(3)

            # 4. Recover Year from Filename
            # Your naming convention is 's1_STATE_YEAR.csv'
            if 'year' not in df.columns:
                filename = os.path.basename(f)
                parts = filename.replace('.csv', '').split('_')
                # Check the last part for a 4-digit year
                if parts[-1].isdigit() and len(parts[-1]) == 4:
                    df['year'] = int(parts[-1])

            # 5. Add to Stack
            all_rows.append(df)

        except Exception as e:
            # Silent fail for corrupt individual files, just skip them
            pass

    # 3. GRAND MERGE
    if all_rows:
        # Concatenate all years/states
        master_df = pd.concat(all_rows, axis=0, ignore_index=True, sort=False)

        # Final ID Polish (Ensure 5-digit string '01001')
        if 'GEOID' in master_df.columns:
            master_df['GEOID'] = master_df['GEOID'].astype(str).str.split('.').str[0].str.zfill(5)

        # Sort by ID and Year
        sort_cols = [c for c in ['GEOID', 'year'] if c in master_df.columns]
        if sort_cols:
            master_df = master_df.sort_values(sort_cols)

        # 4. Save
        out_path = os.path.join(DEST_DIR, 'MASTER_Sentinel1_vFinal.csv')
        master_df.to_csv(out_path, index=False)

        print(f"✅ Saved: {out_path}")
        print(f"   Shape: {master_df.shape} (Rows, Cols)")

        # Validation Stats
        if 'year' in master_df.columns:
            years = sorted(master_df['year'].dropna().unique().astype(int))
            print(f"   Years Covered: {min(years)} to {max(years)}")
            print(f"   Columns Included: {len(master_df.columns)}")

if __name__ == "__main__":
    process_sentinel1_master()

GEE Data merging and preporcessing code begings here Merging the GEE datsets (Individual csv to a master csv)

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION & CLASS MAPPINGS
# ==========================================
SOURCE_DIR = '/content/drive/MyDrive/Agricultural_RS_LE_2025'
DEST_DIR = os.path.join(SOURCE_DIR, 'master csv')

if not os.path.exists(DEST_DIR):
    os.makedirs(DEST_DIR)

# --- CLASS DEFINITIONS (The Decoder Rings) ---
CLASS_MAPS = {
    'Cropland_USDA': {
        '1': 'Corn', '2': 'Cotton', '3': 'Rice', '4': 'Sorghum', '5': 'Soybeans',
        '6': 'Sunflower', '10': 'Peanuts', '11': 'Tobacco', '12': 'Sweet Corn',
        '21': 'Barley', '23': 'Spring Wheat', '24': 'Winter Wheat', '28': 'Oats',
        '36': 'Alfalfa', '37': 'Other Hay', '44': 'Other Crops', '61': 'Fallow',
        '111': 'Open Water', '121': 'Dev_Open', '122': 'Dev_Low',
        '123': 'Dev_Med', '124': 'Dev_High', '141': 'Forest_Deciduous',
        '142': 'Forest_Evergreen', '143': 'Forest_Mixed', '152': 'Shrubland',
        '176': 'Grassland', '190': 'Wetlands_Woody', '195': 'Wetlands_Herbaceous'
    },
    'Dynamic_World': {
        '0': 'Water', '1': 'Trees', '2': 'Grass', '3': 'FloodedVeg',
        '4': 'Crops', '5': 'ShrubScrub', '6': 'Built', '7': 'Bare', '8': 'SnowIce'
    },
    'Water_JRC': {
        '1': 'NotWater', '2': 'Seasonal', '3': 'Permanent'
    }
}

# ==========================================
# 2. PARSING ENGINE
# ==========================================
def parse_histogram_str(text):
    """Robustly parses string '{k=v, ...}' or JSON into a python dict."""
    if pd.isna(text) or str(text).strip() in ['', '{}', 'nan']:
        return {}

    # 1. Fast Path: Standard JSON
    if ':' in str(text) and '{' in str(text):
        try:
            return ast.literal_eval(str(text))
        except: pass

    # 2. Robust Path: GEE "key=value" format
    clean = str(text).replace('{', '').replace('}', '')
    if not clean: return {}

    # Regex handles scientific notation (e.g. 1.2e-5)
    pattern = r'([\d\w\.\-\+]+)\s*=\s*([\d\w\.\-\+eE]+)'
    matches = re.findall(pattern, clean)

    data = {}
    for k, v in matches:
        try:
            # Clean Key: Convert "1.0" -> "1" (Int) for cleaner mapping
            key_num = float(k)
            key = str(int(key_num)) if key_num.is_integer() else str(k)

            data[key] = float(v)
        except: continue
    return data

# ==========================================
# 3. GENERIC PROCESSING (One Code to Rule Them All)
# ==========================================
def process_sensor_master(sensor_name, file_pattern):
    print(f"\n🚜 Processing {sensor_name.upper()}...")

    files = glob.glob(os.path.join(SOURCE_DIR, file_pattern))
    if not files:
        print(f"   ⚠️ No files found for {sensor_name}")
        return

    all_rows = []

    # Get specific mapping for this sensor (if any)
    sensor_map = CLASS_MAPS.get(sensor_name, {})

    for f in tqdm(files, desc=f"Reading {sensor_name}"):
        try:
            # 1. Load File
            df = pd.read_csv(f, on_bad_lines='skip', low_memory=False)

            # 2. Metadata Cleanup
            if 'system:index' in df.columns:
                df = df.drop(columns=['system:index', '.geo', 'Unnamed: 0'], errors='ignore')

            # Ensure GEOID is padded string
            if 'GEOID' not in df.columns and 'STATEFP' in df.columns:
                 df['GEOID'] = df['STATEFP'].astype(str).str.zfill(2) + \
                               df['COUNTYFP'].astype(str).str.zfill(3)

            # Recover Year if missing
            if 'year' not in df.columns:
                parts = os.path.basename(f).replace('.csv','').split('_')
                if parts[-1].isdigit() and len(parts[-1]) == 4:
                    df['year'] = int(parts[-1])

            # 3. HISTOGRAM EXPLOSION
            # Find any column ending in 'histogram'
            hist_cols = [c for c in df.columns if 'histogram' in c.lower()]

            if hist_cols:
                for h_col in hist_cols:
                    # Skip Dynamic World "Confidence" histograms (Probability noise)
                    # We only want the Class Counts (Integer keys)
                    sample = df[h_col].dropna().iloc[0] if not df[h_col].dropna().empty else ""
                    if '0.1' in str(sample) and sensor_name == 'Dynamic_World':
                        df = df.drop(columns=[h_col]) # Drop noise
                        continue

                    # Parse
                    parsed_series = df[h_col].apply(parse_histogram_str)

                    # Explode
                    exploded = pd.json_normalize(parsed_series)
                    exploded.index = df.index

                    # --- INTELLIGENT RENAMING ---
                    new_names = {}
                    for col in exploded.columns:
                        # col is likely "1", "5", etc.
                        if col in sensor_map:
                            # Map "1" -> "Corn"
                            name_suffix = sensor_map[col]
                        else:
                            # Fallback "Class_1"
                            name_suffix = f"Class_{col}"

                        new_names[col] = f"{sensor_name}_{name_suffix}_pct"

                    exploded = exploded.rename(columns=new_names)

                    # --- CRITICAL: ZERO FILLING (LOCAL ONLY) ---
                    # Only fill 0s for crops *within this file*.
                    # If this file exists, we know missing keys = 0 pixels.
                    exploded = exploded.fillna(0)

                    # Normalize to Percentage
                    row_sums = exploded.sum(axis=1)
                    row_sums[row_sums == 0] = 1
                    exploded = exploded.div(row_sums, axis=0)

                    # Merge & Clean
                    df = pd.concat([df, exploded], axis=1)
                    df = df.drop(columns=[h_col])

            all_rows.append(df)

        except Exception as e:
            pass

    # 4. GRAND MERGE (PRESERVING NANs)
    if all_rows:
        # Stack all chunks
        master_df = pd.concat(all_rows, axis=0, ignore_index=True, sort=False)

        # --- CRITICAL: VACANT CELL HANDLING ---
        # We do NOT run master_df.fillna(0) globally.
        # This ensures that if a Year/County didn't exist in the input files,
        # its columns remain NaN (Vacant), indicating "No Data Collected".

        # Final ID Polish
        if 'GEOID' in master_df.columns:
            master_df['GEOID'] = master_df['GEOID'].astype(str).str.split('.').str[0].str.zfill(5)

        # Sort
        sort_cols = [c for c in ['GEOID', 'year'] if c in master_df.columns]
        if sort_cols:
            master_df = master_df.sort_values(sort_cols)

        # Save
        out_path = os.path.join(DEST_DIR, f'MASTER_{sensor_name}_vFinal.csv')
        master_df.to_csv(out_path, index=False)
        print(f"✅ Saved: {out_path}")
        print(f"   Shape: {master_df.shape} (Rows, Cols)")

        # Audit
        if 'year' in master_df.columns:
            years = sorted(master_df['year'].dropna().unique().astype(int))
            print(f"   Years: {min(years)} to {max(years)}")

# ==========================================
# 4. EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    # Define datasets to process
    DATASETS = {
        'Cropland_USDA': 'cropland_usda_*.csv',
        'Dynamic_World': 'dynamic_world_*.csv',
        'Water_JRC':     'water_jrc_*.csv',
        'Landsat':       'landsat89_*.csv',
        'Sentinel2':     's2_*.csv',
        'MODIS_NDVI':    'modis_ndvi_*.csv',
        'MODIS_LST':     'modis_lst_*.csv',
        'DEM':           'dem_*.csv',
        'Soil':          'soil_texture_*.csv'
    }

    for name, pattern in DATASETS.items():
        process_sensor_master(name, pattern)